In [4]:
from aiida.engine import run_get_node
from topworkchain import PrototypeTopWorkChain
from aiida.orm import FolderData
from aiida import load_profile
load_profile()

Profile<uuid='52dacf545d5c4d089ba5e0dd6267dcf0' name='presto'>

In [5]:
#Run the model
results, node = run_get_node(PrototypeTopWorkChain)
#Load the FolderData node containing the compressed cube files (a folder of files stored on the AiiDA database.)
cube_folder = results["cube_folder"]
#Load the Dict node containing all of the relevant details of the transitions.
transition_info_node = results["transition_info"]
#Convert to a standard Python dict.
transition_info = transition_info_node.get_dict()


04/14/2026 09:09:12 PM <2170222> aiida.broker.rabbitmq: [WARNING] RabbitMQ v3.12.1 is not supported and will cause unexpected problems!
04/14/2026 09:09:12 PM <2170222> aiida.broker.rabbitmq: [WARNING] It can cause long-running workflows to crash and jobs to be submitted multiple times.
04/14/2026 09:09:12 PM <2170222> aiida.broker.rabbitmq: [WARNING] See https://github.com/aiidateam/aiida-core/wiki/RabbitMQ-version-to-use for details.


uuid: bd23b147-64ac-4e07-8dfa-5422cde30c26 (pk: 14476) (subworkchains.OrcaWorkChain)
uuid: 4cb803c6-b73f-4809-aee1-74b70cd930d6 (pk: 14499)
uuid: a2962d5d-21d0-4ef7-8fcc-dbae3f5a4202 (pk: 14513)
uuid: 54ba014d-10a1-4888-a907-7162844ad4e4 (pk: 14528)
uuid: 91209d69-a6fd-4cec-bc77-4398c2505a74 (pk: 14542)
uuid: 7dd444c2-edd6-4e96-a839-7676a8cf26dd (pk: 14557)
uuid: 181b10fc-213b-4176-89d5-feb0615fe8f3 (pk: 14571)
uuid: 7e771163-5456-464d-b6e8-f10cfd317f6b (pk: 14585)
uuid: 2bcae540-9843-40f5-af8d-0caab852eba8 (pk: 14599)
uuid: c7cb22ae-e578-44ed-900f-5d6605e3754d (pk: 14614)
uuid: 3242d895-1085-4364-83f3-6f1f9a33d0e5 (pk: 14628)
uuid: 787428e1-9c94-4710-b233-a20c87fd8abc (pk: 14643)
uuid: a70cd204-96fd-428f-9a25-d98f92476cdc (pk: 14657)
uuid: 07a10228-1440-4cb1-ab8e-40e4b5724f23 (pk: 14672)
uuid: ecb5dc59-a85e-477b-97f4-44fdd3530d0a (pk: 14686)
uuid: 0802fb57-590e-460a-986c-8c6206638182 (pk: 14700)
uuid: 74d47cae-7d02-4a04-9bb4-1036469b01ac (pk: 14714)
uuid: db203df5-ccf6-402b-88ea-930a1

In [6]:
import nglview as nv
import ipywidgets as widgets
import traitlets as tl
import ase.io
import io
import os
from ase.build import molecule
import tempfile
from pathlib import Path
from ipywidgets import Layout

In [17]:
# Create an output directory for the temporary cube files
output_dir = Path("extracted_cubes")
output_dir.mkdir(exist_ok=True)

# Create a list of the cube files in the data node
names = cube_folder.list_object_names()

# Read in the cube files and write them to temporary files
for name in names:
    with cube_folder.open(name, mode="rb") as file_in:
        with open(f"{output_dir}/{name}.cube", "wb") as file_out:
            file_out.write(file_in.read())

# cube_names = sorted([name for name in os.listdir("./extracted_cubes") if name.endswith(".cube")])


# # Create a list of the paths to the temporary cube files
# CUBE_PATHS = [f"{output_dir}/{name}.cube" for name in names]

specific_excitations = list(transition_info.keys())

#Function to change the options available in the transition_dropdown.
def transition_update(change):
    #Isolate specific relevant transitions.
    specific_transitions = transition_info[excitation_dropdown.value]
    #Adjust format of transitions.
    specific_transitions2 = []
    for each_transition in specific_transitions:
        specific_transitions2.append((((each_transition[0])[0]+" -> "+(each_transition[0])[1]+" ("+each_transition[1]+")"), each_transition))
    transition_dropdown.options = specific_transitions2

# Create dropdown widgets for selecting the cube files
excitation_dropdown = widgets.Dropdown(
    options=specific_excitations,
    value=specific_excitations[0] if specific_excitations else None,
    description="Excitation (s)"
 )

transition_dropdown = widgets.Dropdown(
    description="Transition"
 )

#Run transition_update when excitation_dropdown is clicked.
excitation_dropdown.observe(transition_update, names="value")

#Initialise the transition_dropdown with inital options and value.
transition_update({"new": excitation_dropdown.value})
transition_dropdown.value = transition_dropdown.options[0][1] if transition_dropdown.options else None

# Initialize views with current dropdown selections
CUBE_PATH_1 = f"./extracted_cubes/{"s"+excitation_dropdown.value+"."+((transition_dropdown.value[0])[0])[:-1]+".cube"}"
CUBE_PATH_2 = f"./extracted_cubes/{"s"+excitation_dropdown.value+"."+((transition_dropdown.value[0])[1])[:-1]+".cube"}"

# print(CUBE_PATH_1)
# print(CUBE_PATH_2)
# Create NGL views for the selected cube files
molecule1 = nv.show_ase(ase.io.read(CUBE_PATH_1))
molecule2 = nv.show_ase(ase.io.read(CUBE_PATH_2))
NTO_1 = molecule1.add_component(CUBE_PATH_1, ext="cube")
NTO_2 = molecule2.add_component(CUBE_PATH_2, ext="cube")


# Create sliders for adjusting the isovalue (not visible in the NGL view, but used to set the isovalue of the surfaces)
positive_level = widgets.FloatLogSlider(
    value=1e-3,
    base=10,
    min=-5,
    max=-1,
    step=0.1,
    description="+ isovalue",
    readout_format=".1e",
 )
negative_level = widgets.FloatLogSlider(
    value=1e-3,
    base=10,
    min=-5,
    max=-1,
    step=0.1,
    description="- isovalue",
    readout_format=".1e",
 )

# List of colour blind friendly colours
colours = ['#377eb8', '#ff7f00', '#4daf4a',
                  '#f781bf', '#a65628', '#984ea3',
                  '#999999', '#e41a1c', '#dede00']

# To store colour buttons
pos_palette_buttons = []
neg_palette_buttons = []

def set_positive_colour(colour):
    selected_colours["positive"] = colour
    redraw_surfaces()

def set_negative_colour(colour):
    selected_colours["negative"] = colour
    redraw_surfaces()

# Create buttons for each colour for both palettes
for c in colours:
    b = widgets.Button(
        layout=widgets.Layout(width='30px', height='30px'),
        style={'button_color': c}
    )
    b.on_click(lambda _, c=c: set_positive_colour(c))
    pos_palette_buttons.append(b)

for c in colours:
    b = widgets.Button(
        layout=widgets.Layout(width='30px', height='30px'),
        style={'button_color': c}
    )
    b.on_click(lambda _, c=c: set_negative_colour(c))
    neg_palette_buttons.append(b)

pos_palette = widgets.HBox(pos_palette_buttons)
neg_palette = widgets.HBox(neg_palette_buttons)

# Preset colours 
selected_colours = {
    "positive": "#377eb8",
    "negative": "#e41a1c"
}

status = widgets.HTML()

# Define functions to redraw the surfaces when the sliders or colour scheme are changed
def redraw_NTO_1(_=None):
    NTO_1.clear()
    NTO_1.add_surface(
        color=selected_colours["positive"],
        isolevelType="value",
        isolevel=float(positive_level.value),
        opacity=0.5,
    )
    NTO_1.add_surface(
        color=selected_colours["negative"],
        isolevelType="value",
        isolevel=-float(negative_level.value),
        opacity=0.5,
    )
def redraw_NTO_2(_=None):
    NTO_2.clear()
    NTO_2.add_surface(
        color=selected_colours["positive"],
        isolevelType="value",
        isolevel=float(positive_level.value),
        opacity=0.5,
    )
    NTO_2.add_surface(
        color=selected_colours["negative"],
        isolevelType="value",
        isolevel=-float(negative_level.value),
        opacity=0.5,
    )
def redraw_surfaces(_=None):
    redraw_NTO_1()
    redraw_NTO_2()

def update_NTO_1(_=None):
    #print("s"+excitation_dropdown.value+"."+((transition_dropdown.value[0])[0])[:-1]+".cube")
    global NTO_1
    NTO_1.clear()
    NTO_1 = molecule1.add_component(f"./extracted_cubes/{"s"+excitation_dropdown.value+"."+((transition_dropdown.value[0])[0])[:-1]+".cube"}", ext="cube")  # Add path!
    redraw_NTO_1()
    
def update_NTO_2(_=None):
    #print("s"+excitation_dropdown.value+"."+((transition_dropdown.value[0])[1])[:-1]+".cube")
    global NTO_2
    NTO_2.clear()
    NTO_2 = molecule2.add_component(f"./extracted_cubes/{"s"+excitation_dropdown.value+"."+((transition_dropdown.value[0])[1])[:-1]+".cube"}", ext="cube")  # Add path!
    redraw_NTO_2()

# Set up observers to redraw the surfaces when the sliders or colour scheme are changed
for control in [positive_level, negative_level]:
    control.observe(redraw_surfaces, names="value")

excitation_dropdown.observe(update_NTO_1, names="value")
transition_dropdown.observe(update_NTO_1, names="value")
excitation_dropdown.observe(update_NTO_2, names="value")
transition_dropdown.observe(update_NTO_2, names="value")

redraw_surfaces()

# Create a box to contain the colour palettes
palette_box = widgets.VBox([
    widgets.HTML("<b>Positive colour</b>"),
    pos_palette,
    widgets.HTML("<b>Negative colour</b>"),
    neg_palette
    
])

# Create drop down for displaying colours
accordion = widgets.Accordion(children=[palette_box])
accordion.set_title(0, 'Select colours')
accordion.selected_index = None  # start closed
accordion.layout = Layout(width="25%")

# Arrange the widgets in a layout
controls = widgets.VBox(
    [
        widgets.HTML("<b>Cube isosurfaces</b>"),
        accordion,
        status,
    ],
 )
molecule1_box = widgets.VBox([excitation_dropdown, molecule1], layout= Layout(width="100%", flex="1 1 50%", min_width="0"))
molecule2_box = widgets.VBox([transition_dropdown, molecule2], layout= Layout(width="100%", flex="1 1 50%", min_width="0"))
molecule_box = widgets.HBox([molecule1_box, molecule2_box], layout=Layout(width="100%", flex="1 1 100%"))

widgets.VBox([controls, molecule_box], layout = Layout(width="100%", flex="1 1 100%"))


0.70386215


In [6]:
test = nv.show_ase(ase.io.read("./extracted_cubes/s6.14.cube"))
test.add_component("./extracted_cubes/s6.14.cube", ext="cube", default_representation=False)

test.display()

NGLWidget()

[[['14a', '15a'], '0.70386215'], [['13a', '16a'], '0.16175032'], [['12a', '17a'], '0.13088409']]
